<a href="https://colab.research.google.com/github/mailtoshikharchauhan-lab/Bottle-Quality-Inspection/blob/main/Bottle_Quality_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 65.2 MB/s eta 0:00:00


In [2]:
from ultralytics import YOLO
import os
import shutil

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
!pip install kagglehub

In [4]:
import kagglehub

path = kagglehub.dataset_download(
    "youssefeldemerdashh/big-cola-bottles-dataset-for-yolo-quality-control"
)

print(path)

Using Colab cache for faster access to the 'big-cola-bottles-dataset-for-yolo-quality-control' dataset.
/kaggle/input/big-cola-bottles-dataset-for-yolo-quality-control


In [5]:
import os

for root, dirs, files in os.walk(path):
    print(root)

/kaggle/input/big-cola-bottles-dataset-for-yolo-quality-control
/kaggle/input/big-cola-bottles-dataset-for-yolo-quality-control/obj_train_data


In [8]:
import os

dataset_path = os.path.join(path, "obj_train_data")

print(os.listdir(dataset_path))

['20260413_031929.jpg', '20260413_035424.jpg', '20260413_035534.jpg', '20260413_034501.jpg', '20260413_035519.jpg', '20260413_040406.txt', '20260413_024636.jpg', '20260413_025428.jpg', '20260413_025538.txt', '20260413_034458.txt', '20260413_025457.txt', '20260413_035247.jpg', '20260413_041145.jpg', '20260413_040849.txt', '20260413_035238.txt', '20260413_214301.txt', '20260413_212150.txt', '20260413_035426.jpg', '20260413_034739.jpg', '20260413_040602.jpg', '20260413_034849.jpg', '20260413_035526.jpg', '20260413_032043.jpg', '20260413_025502.txt', '20260413_034840.txt', '20260413_040406.jpg', '20260413_032035.jpg', '20260413_031407.txt', '20260413_035018.txt', '20260413_040230.jpg', '20260413_035027.jpg', '20260413_041145.txt', '20260413_035158.txt', '20260413_034856.txt', '20260413_040245.jpg', '20260413_035231.txt', '20260413_034800.jpg', '20260413_031358.txt', '20260413_040131.jpg', '20260413_031946.txt', '20260413_035022.jpg', '20260413_035509.jpg', '20260413_211908.jpg', '20260413_

In [9]:
for root, dirs, files in os.walk(dataset_path):
    print(root)
    print("Folders:", dirs)
    print("Files:", files[:5])
    print("-"*60)

/kaggle/input/big-cola-bottles-dataset-for-yolo-quality-control/obj_train_data
Folders: []
Files: ['20260413_031929.jpg', '20260413_035424.jpg', '20260413_035534.jpg', '20260413_034501.jpg', '20260413_035519.jpg']
------------------------------------------------------------


In [10]:
import os

files = os.listdir(dataset_path)

for f in files:
    if f.endswith(".txt") and f.lower() in ["classes.txt", "obj.names"]:
        print(f)

In [11]:
import os

base = "/content/bottle_dataset"

folders = [
    "train/images",
    "train/labels",
    "valid/images",
    "valid/labels",
    "test/images",
    "test/labels"
]

for folder in folders:
    os.makedirs(os.path.join(base, folder), exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [12]:
import os
import random
import shutil

source = dataset_path

images = [f for f in os.listdir(source) if f.endswith(".jpg")]

random.seed(42)
random.shuffle(images)

n = len(images)

train_end = int(n * 0.7)
valid_end = int(n * 0.9)

train = images[:train_end]
valid = images[train_end:valid_end]
test = images[valid_end:]

In [13]:
def copy_files(file_list, split):
    for img in file_list:
        txt = img.replace(".jpg", ".txt")

        shutil.copy(
            os.path.join(source, img),
            f"/content/bottle_dataset/{split}/images/{img}"
        )

        if os.path.exists(os.path.join(source, txt)):
            shutil.copy(
                os.path.join(source, txt),
                f"/content/bottle_dataset/{split}/labels/{txt}"
            )

copy_files(train, "train")
copy_files(valid, "valid")
copy_files(test, "test")

print("Dataset split complete!")

Dataset split complete!


In [14]:
data_yaml = """
train: /content/bottle_dataset/train/images
val: /content/bottle_dataset/valid/images
test: /content/bottle_dataset/test/images

nc: 5

names:
  0: Perfect_bottle
  1: empty_bottle
  2: no_cap
  3: no_label
  4: crooked_cap
"""

with open("/content/bottle_dataset/data.yaml", "w") as f:
    f.write(data_yaml)

print("data.yaml created successfully!")

data.yaml created successfully!


In [15]:
import os

print("Train Images :", len(os.listdir("/content/bottle_dataset/train/images")))
print("Train Labels :", len(os.listdir("/content/bottle_dataset/train/labels")))

print("Valid Images :", len(os.listdir("/content/bottle_dataset/valid/images")))
print("Valid Labels :", len(os.listdir("/content/bottle_dataset/valid/labels")))

print("Test Images :", len(os.listdir("/content/bottle_dataset/test/images")))
print("Test Labels :", len(os.listdir("/content/bottle_dataset/test/labels")))

Train Images : 263
Train Labels : 263
Valid Images : 76
Valid Labels : 76
Test Images : 38
Test Labels : 38


In [16]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

In [17]:
results = model.train(
    data="/content/bottle_dataset/data.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    workers=2,
    patience=5,
    device=0,
    project="BottleQuality",
    name="BottleQC"
)

Ultralytics 8.4.87 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/bottle_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=BottleQC, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/BottleQuality/BottleQC/weights/best.pt")

results = model.predict(
    source="/content/bottle_dataset/test/images",
    conf=0.25,
    save=True,
    show_labels=True,
    show_conf=True
)